# Update and Delete (V2)

This notebook shows how to change and remove data that was already created in V2:
- Update fields of test results, steps, and measurements, one at a time or in a batch.
- Add, replace, and remove the conditions of a step.
- Link and unlink software items, hardware items, and test adapters.
- Delete measurements, steps, and test results, with and without `force`.

Not every field can be changed. For example, a measurement's name and value cannot be changed after it is created.

## Prerequisites

- NI Measurement Data Services running in V2 mode
- Python environment with the `ni.datastore` package (see `../README.md`)

In [ ]:
# Perform example-specific setup. This is not needed when writing production code.
# Set MDS_V2_GRPC_TARGET (for example, "[::1]:59606") to connect to a specific V2 service.
# When it is not set, the clients use discovery to find the service.
import os

import grpc

grpc_target = os.environ.get("MDS_V2_GRPC_TARGET", "")
# Remove gRPC's 4 MB receive limit so that large values, such as long waveforms, can be read.
grpc_options = [("grpc.max_receive_message_length", -1)]
grpc_channel = grpc.insecure_channel(grpc_target, options=grpc_options) if grpc_target else None

from ni.datastore.data.v2 import DataStoreClient
from ni.datastore.metadata import MetadataStoreClient

metadata_store_client = MetadataStoreClient(grpc_channel=grpc_channel)
data_store_client = DataStoreClient(grpc_channel=grpc_channel)

## Create the Example Data

In [ ]:
from ni.datastore.data.v2 import Condition, Outcome, Status, Step, TestResult
from ni.datastore.metadata import SoftwareItem

software_item_id = metadata_store_client.create_software_item(
    SoftwareItem(product="Example Driver", version="1.0.0")
)

# Create methods return the created objects. Keep them to update and delete them later.
test_result = data_store_client.create_test_result(
    TestResult(name="Update and Delete Example", status=Status.RUNNING)
)
step = data_store_client.create_step(
    Step(
        name="Example Step",
        test_result_id=test_result.id,
        status=Status.RUNNING,
        conditions=[
            Condition("Ambient Temperature", 23.0, condition_type="Environment", units="degC"),
            Condition("Test Fixture", "FIX-001", condition_type="Setup"),
        ],
    )
)
measurements = [
    data_store_client.create_measurement(f"Example Measurement {index}", float(index), step.id)
    for index in range(3)
]
print(f"Created Test Result: {test_result.id}")
print(f"Created Step: {step.id}")
print(f"Created Measurements: {[measurement.id for measurement in measurements]}")

## Update Fields

There are three ways to update an object. Each returns the changed object.
- Pass the object, or its ID, with the fields to change as keyword arguments. Only those fields change.
- Pass an update builder, such as `TestResultUpdateBuilder`. Each `with_` call selects one field to change.
- Change the object and pass it on its own. Every field that can be updated is sent, so a field that is empty is cleared.

`status` describes how far the work has progressed, and `outcome` describes the result. They are separate, so a test can be `DONE` and `FAILED`.

In [ ]:
from ni.datastore.data.v2 import TestResultUpdateBuilder

# Keyword fields: only the fields that are passed change.
step = data_store_client.update_step(
    step, status=Status.DONE, outcome=Outcome.FAILED, notes="Output was out of range."
)
print(f"step: {step.status.name}, {step.outcome.name}, notes='{step.notes}'")

# Update builder: each with_ call selects one field to change.
test_result = data_store_client.update_test_result(
    TestResultUpdateBuilder(test_result)
    .with_status(Status.DONE)
    .with_outcome(Outcome.FAILED)
    .with_link("https://example.com/runs/1")
)
print(f"test result: {test_result.status.name}, {test_result.outcome.name}, link={test_result.link}")

# Whole object: change the returned object, then send it back.
measurements[0].notes = "Rechecked by operator."
measurements[0] = data_store_client.update_measurement(measurements[0])
print(f"measurement notes: '{measurements[0].notes}'")

assert test_result.status == Status.DONE
assert test_result.outcome == Outcome.FAILED

Update multiple items in one atomic call. Pass update builders or objects, or a dictionary of fields keyed by ID:

In [ ]:
from ni.datastore.data.v2 import MeasurementUpdateBuilder

measurements = list(
    data_store_client.batch_update_measurements(
        [
            MeasurementUpdateBuilder(measurements[0]).with_outcome(Outcome.PASSED),
            MeasurementUpdateBuilder(measurements[1]).with_outcome(Outcome.FAILED),
            MeasurementUpdateBuilder(measurements[2])
            .with_outcome(Outcome.PASSED)
            .with_notes("Borderline."),
        ]
    )
)
for measurement in measurements:
    print(f"{measurement.name}: {measurement.outcome.name}")

## Add, Replace, and Remove Conditions

Conditions cannot be changed in place. Adding a condition with an existing name replaces it.

In [ ]:
step = data_store_client.add_condition(
    step.id, Condition("Relative Humidity", 45.0, condition_type="Environment", units="%")
)
step = data_store_client.add_condition(
    step.id, Condition("Ambient Temperature", 24.5, condition_type="Environment", units="degC")
)
step = data_store_client.remove_condition(step.id, "Test Fixture")

for condition in step.conditions:
    print(f"{condition.name} = {condition.value} {condition.units}")

condition_names = {condition.name for condition in step.conditions}
assert condition_names == {"Ambient Temperature", "Relative Humidity"}

## Link and Unlink Metadata

Link a software item to the test result first. Then a measurement of that test result can also link to it. Each measurement keeps its own selection: linking an item to the test result does not link it to the test result's measurements.

To unlink, reverse the order: unlink the item from the measurement, then from the test result. An item cannot be unlinked from a test result while any of its measurements still references it.

In [ ]:
test_result = data_store_client.add_test_result_metadata(
    test_result.id, software_item_ids=[software_item_id]
)
print(f"test result software items: {list(test_result.software_item_ids)}")

measurement = data_store_client.add_measurement_metadata(
    measurements[0].id, software_item_ids=[software_item_id]
)
print(f"measurement software items: {list(measurement.software_item_ids)}")

measurement = data_store_client.remove_measurement_metadata(
    measurements[0].id, software_item_ids=[software_item_id]
)
print(f"measurement software items after unlinking: {list(measurement.software_item_ids)}")

test_result = data_store_client.remove_test_result_metadata(
    test_result.id, software_item_ids=[software_item_id]
)
print(f"test result software items after unlinking: {list(test_result.software_item_ids)}")
assert list(test_result.software_item_ids) == []

## Delete

Delete methods accept the object or its ID. Deleting a measurement removes it. Getting it afterward fails with `NOT_FOUND`.

In [ ]:
import grpc

data_store_client.delete_measurement(measurements[2])

try:
    data_store_client.get_measurement(measurements[2].id)
except grpc.RpcError as e:
    print(f"{e.code().name}: {e.details()}")

A step or test result that still owns data cannot be deleted unless you pass `force=True`. This protects you from deleting data by accident.

In [ ]:
try:
    data_store_client.delete_step(step)
except grpc.RpcError as e:
    print(f"{e.code().name}: {e.details()}")

try:
    data_store_client.delete_test_result(test_result)
except grpc.RpcError as e:
    print(f"{e.code().name}: {e.details()}")

Delete multiple items in one atomic call, or use `force=True` to also delete everything the item owns:

In [ ]:
data_store_client.batch_delete_measurements(measurements[:2])
data_store_client.delete_step(step)  # No measurements are left, so force is not needed.
data_store_client.delete_test_result(test_result)

# Create two more test results, each with a step, and delete them together.
extra_test_results = data_store_client.batch_create_test_results(
    [
        TestResult(name="Delete Example A", steps=[Step(name="Step A")]),
        TestResult(name="Delete Example B", steps=[Step(name="Step B")]),
    ]
)
data_store_client.batch_delete_test_results(extra_test_results, force=True)

remaining = data_store_client.query_test_results(
    "$filter=Name eq 'Update and Delete Example' or startswith(Name,'Delete Example')"
)
print(f"Remaining example test results: {len(remaining)}")
assert len(remaining) == 0

## Clean up

In [ ]:
metadata_store_client.close()
data_store_client.close()

# Perform example-specific cleanup. This is not needed when writing production code.
if grpc_channel is not None:
    grpc_channel.close()